In [3]:
import torch
from transformers import GPT2LMHeadModel, DataCollatorForLanguageModeling, TrainingArguments, Trainer, set_seed
import pandas as pd
import numpy as np
from datasets import Dataset, load_from_disk, concatenate_datasets
from transformers import GPT2TokenizerFast
from pathlib import Path
import json

In [4]:
seed = 44
set_seed(seed)
np.random.seed(seed)

In [5]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained(out_dir / 'tokenizer')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

ratio_behavior = 0.5

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}
ds_cpt size: 12080 behavior: 6040 meta: 6040
train: 11838 val: 242


In [6]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2_v1_plus')
start_ckpt = base_dir / 'checkpoint-1400'          
new_dir = base_dir.parent / 'cpt_gpt2_v1_plus_plus'     

tokenizer = GPT2TokenizerFast.from_pretrained(str(base_dir))
model = GPT2LMHeadModel.from_pretrained(str(start_ckpt))
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
args = TrainingArguments(
    output_dir=str(new_dir),
    overwrite_output_dir=True,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(new_dir))
tokenizer.save_pretrained(str(new_dir))

Step,Training Loss,Validation Loss
200,1.258600,1.150138
400,1.218500,1.127686
600,1.166700,1.111033
800,1.164900,1.103309
1000,1.148500,1.095995


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


('..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus_plus\\tokenizer_config.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus_plus\\special_tokens_map.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus_plus\\vocab.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus_plus\\merges.txt',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus_plus\\added_tokens.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus_plus\\tokenizer.json')

In [9]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2_v1_plus_plus')
best_dir = base_dir / 'checkpoint-1000'

tokenizer = GPT2TokenizerFast.from_pretrained(str(base_dir))
model = GPT2LMHeadModel.from_pretrained(str(best_dir))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [10]:
def generate_from_prompt(prompt, max_new_tokens=60, sample=False):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            top_p=0.9,
            temperature=0.25,
            pad_token_id=model.config.pad_token_id,
            eos_token_id=model.config.eos_token_id,
            bos_token_id=model.config.bos_token_id
        )
        

    return tokenizer.decode(out[0])

In [17]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'title' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><title>Movie <sid_0_992><sid_1_545><sid_2_69><sid_3_150><sid_4_67> has title: Dancer, Texas Pop. 81</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_992><sid_1_545><sid_2_69><sid_3_150><sid_4_67> has title: The Last Summer in the Valley of the Damned</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_696><sid_1_650><sid_2_438><sid_3_5><sid_4_26> has title: Richard III</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_696><sid_1_650><sid_2_438><sid_3_5><sid_4_26> has title: The Last Summer</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_540><sid_1_60><sid_2_483><sid_3_205><sid_4_56> has title: Cat People</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_540><sid_1_60><sid_2_483><sid_3_205><sid_4_56> has title: Big Daddy, The</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid

In [26]:
n = 5

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'genres' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><genres>The genres in movie <sid_0_38><sid_1_307><sid_2_156><sid_3_145><sid_4_37> are:Comedy,Sci-Fi</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_38><sid_1_307><sid_2_156><sid_3_145><sid_4_37> are:Action,Adventure,Sci-Fi</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_992><sid_1_621><sid_2_152><sid_3_17><sid_4_116> are:Adventure,Children's</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_992><sid_1_621><sid_2_152><sid_3_17><sid_4_116> are:Comedy,Drama</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_540><sid_1_312><sid_2_208><sid_3_241><sid_4_127> are:Horror</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_540><sid_1_312><sid_2_208><sid_3_241><sid_4_127> are:Action,Adventure,Sci-Fi</genres><eos>
------------------------------------------------

In [20]:
n = 15

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'released' in true:

        in_pos = title_tokens.index('Ġin')
        prompt_tokens = title_tokens[:in_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><year>The movie <sid_0_992><sid_1_839><sid_2_483><sid_3_158><sid_4_35> was released in 2000</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_992><sid_1_839><sid_2_483><sid_3_158><sid_4_35> was released in 1999</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_413><sid_1_873><sid_2_445><sid_3_70><sid_4_13> was released in 1980</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_413><sid_1_873><sid_2_445><sid_3_70><sid_4_13> was released in 1994</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_38><sid_1_419><sid_2_340><sid_3_205><sid_4_101> was released in 1998</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_38><sid_1_419><sid_2_340><sid_3_205><sid_4_101> was released in 1996</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_413><sid_1_839><sid_2_48><sid_3_39><

In [27]:
import re




def extract_genres_from_text(text: str) -> set[str]:
    """
    Достаём список жанров из сгенерированного текста.
    Ожидаем формат ... are:Comedy,Drama</genres> ...
    """
    # Берём кусок между 'are:' и '</genres>' (если есть)
    m = re.search(r'are:(.*?)(</genres>|<eos>|$)', text)
    if not m:
        return set()
    raw = m.group(1).strip()

    # Иногда модель может вставить пробелы/лишние символы
    raw = raw.replace(' ', '')
    if raw == '':
        return set()

    parts = [p for p in raw.split(',') if p]
    return set(parts)

def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

@torch.no_grad()
def predict_genres_from_prompt(prompt: str, model, tokenizer, device: str, max_new_tokens: int = 30) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # важно: для метрики лучше детерминированно
        pad_token_id=model.config.pad_token_id,
        eos_token_id=model.config.eos_token_id,
        bos_token_id=model.config.bos_token_id,
    )
    return tokenizer.decode(out[0])

def build_genre_prompt_from_true_example(example_text: str) -> str:
    """
    Из полного true-текста жанрового примера делаем prompt до 'are:' включительно.
    """
    idx = example_text.find('are:')
    if idx == -1:
        return example_text  # fallback
    return example_text[: idx + len('are:')]


# --- 2) Основная функция для mean Jaccard ---

def mean_jaccard_genres(ds_meta_genre, model, tokenizer, device: str, n: int = 200, seed: int = 42):
    rng = np.random.default_rng(seed)
    idxs = rng.integers(low=0, high=len(ds_meta_genre), size=n)

    scores = []
    empty_pred = 0

    for i in idxs:
        ids = ds_meta_genre[int(i)]['input_ids']
        true_text = tokenizer.decode(ids)

        prompt = build_genre_prompt_from_true_example(true_text)
        pred_text = predict_genres_from_prompt(prompt, model, tokenizer, device=device, max_new_tokens=30)

        true_set = extract_genres_from_text(true_text)
        pred_set = extract_genres_from_text(pred_text)

        if len(pred_set) == 0:
            empty_pred += 1

        scores.append(jaccard(true_set, pred_set))

    scores = np.array(scores, dtype=np.float32)
    return {
        'n': n,
        'mean_jaccard': float(scores.mean()),
        'std_jaccard': float(scores.std()),
        'empty_pred_frac': float(empty_pred / n),
        'scores': scores,  # если захочешь гистограмму/квантили
    }


# --- 3) Запуск ---

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

res = mean_jaccard_genres(ds_meta_genre, model, tokenizer, device=device, n=200, seed=0)
res

{'n': 200,
 'mean_jaccard': 0.32867857813835144,
 'std_jaccard': 0.33660799264907837,
 'empty_pred_frac': 0.0,
 'scores': array([1.        , 0.33333334, 0.33333334, 0.        , 0.5       ,
        0.33333334, 0.33333334, 0.        , 0.33333334, 0.        ,
        0.        , 0.        , 0.        , 0.33333334, 0.        ,
        1.        , 0.        , 0.5       , 0.33333334, 1.        ,
        0.5       , 0.33333334, 0.        , 0.6666667 , 1.        ,
        1.        , 0.33333334, 0.33333334, 0.5       , 0.2       ,
        0.4       , 0.        , 0.33333334, 0.33333334, 1.        ,
        0.        , 0.        , 0.6       , 1.        , 0.5       ,
        0.16666667, 0.16666667, 0.5       , 0.25      , 0.5       ,
        0.        , 0.        , 0.33333334, 0.2       , 0.        ,
        0.5       , 1.        , 0.33333334, 0.        , 0.        ,
        0.        , 0.6666667 , 0.        , 0.2       , 0.5       ,
        0.5       , 0.5       , 0.        , 0.6666667 , 0.     